# Quadratic program

This notebook shows how to define and solve a small quadratic program using `spacecore` for the spaces and linear maps.

The problem is

$$
\min_x\ \frac12 \langle x, Qx \rangle + \langle q, x \rangle
$$

subject to

$$
A x = b, \qquad x \ge 0.
$$

`spacecore` represents `Q` and `A` as `DenseLinOp` objects. SciPy handles the numerical optimization.

Set `SPACECORE_EXAMPLE_BACKEND=jax` before launching the notebook to run the same code with the JAX backend.

In [ ]:
import os

from scipy.optimize import minimize

from spacecore.backend import Context, JaxOps, NumpyOps
from spacecore.space import DenseVectorSpace
from spacecore.linop import DenseLinOp

backend_name = os.environ.get("SPACECORE_EXAMPLE_BACKEND", "numpy").lower()
if backend_name == "jax":
    # SciPy SLSQP expects float64 arrays at the solver boundary.
    JaxOps.jax.config.update("jax_enable_x64", True)

backend_ops = {"numpy": NumpyOps, "jax": JaxOps}[backend_name]
ctx = Context(backend_ops(), dtype="float64", enable_checks=True)
ops = ctx.ops

## Define the QP data

The decision variable lives in a three-dimensional vector space. The quadratic form and equality constraint are both linear operators between spaces.

In [ ]:
X = DenseVectorSpace((3,), ctx=ctx)
Y = DenseVectorSpace((1,), ctx=ctx)

Q_matrix = ctx.asarray(
    [
        [4.0, 1.0, 0.0],
        [1.0, 2.0, 0.0],
        [0.0, 0.0, 3.0],
    ]
)
A_matrix = ctx.asarray([[1.0, 1.0, 1.0]])

Q = DenseLinOp(Q_matrix, X, X, ctx=ctx)
A = DenseLinOp(A_matrix, X, Y, ctx=ctx)

# This q makes the unconstrained first-order condition compatible with
# x = [0.2, 0.5, 0.3] under the equality constraint.
q = ctx.asarray([-1.3, -1.2, -0.9])
b = ctx.asarray([1.0])

eigenvalues, _ = ops.eigh(Q_matrix)

print("eigenvalues of Q:", eigenvalues)
print("A maps X to Y:", A.dom.shape, "->", A.cod.shape)

## Objective, gradient, and constraints

The operator `Q` supplies the quadratic action and `A` supplies the equality constraint. The adjoint `A.rapply` is used later for a KKT residual check.

In [ ]:
def objective(x_raw):
    x = ctx.asarray(x_raw)
    quadratic_term = 0.5 * X.inner(x, Q.apply(x))
    linear_term = X.inner(q, x)
    return float(ops.real(quadratic_term + linear_term))


def gradient(x_raw):
    x = ctx.asarray(x_raw)
    return Q.apply(x) + q


def equality_residual(x_raw):
    x = ctx.asarray(x_raw)
    return A.apply(x) - b


def equality_jacobian(_x_raw):
    return A_matrix

## Solve

SLSQP handles the nonnegative bounds and the equality constraint. The initial point is feasible.

In [ ]:
x0 = ctx.asarray([1.0 / 3.0, 1.0 / 3.0, 1.0 / 3.0])

result = minimize(
    objective,
    x0,
    jac=gradient,
    method="SLSQP",
    bounds=[(0.0, None)] * X.shape[0],
    constraints={
        "type": "eq",
        "fun": equality_residual,
        "jac": equality_jacobian,
    },
    options={"ftol": 1e-12, "maxiter": 200},
)

x_star = ctx.asarray(result.x)

print("success:", result.success)
print("message:", result.message)
print("x*:", x_star)
print("objective:", objective(x_star))
print("A x*:", A.apply(x_star))

## KKT check

For this example the nonnegative bounds are inactive at the solution. The stationarity residual is

$$
Qx + q + A^T \lambda.
$$

The adjoint action `A.rapply(lambda_eq)` computes `A^T lambda_eq`.

In [ ]:
unconstrained_gradient = Q.apply(x_star) + q
lambda_eq = ops.reshape(-ops.sum(unconstrained_gradient) / X.shape[0], (1,))
stationarity = unconstrained_gradient + A.rapply(lambda_eq)
stationarity_norm = ops.sqrt(ops.sum(ops.abs(stationarity) ** 2))

print("lambda:", lambda_eq)
print("stationarity residual:", stationarity)
print("stationarity norm:", stationarity_norm)
print("equality residual:", equality_residual(x_star))